In [1]:
# Check Test Plan for more details 
# Test ResNet50 model on MNIST dataset
# New Classifier - TCL/TRL Model Our Method
# Optimizer Adam - Default
# No Scheduler
# MNIST dataset -> (3, 192, 192) 
# Pretrained
# Trasfer Learning
# Without Adaptive avg pooling
########################################################

# Add all .py files to path
import sys
sys.path.append('..')

# Import Libraries
from Utils.Accuracy_measures import topk_accuracy
from Utils.Mnist_loader import get_mnist_dataloaders
from Utils.Num_parameter import count_parameters
from Models.Resnet50 import Resnet50
from TRL import TRL


import torchvision.transforms as transforms
from torch import nn
from torch import optim

import time
import torch
import os

In [5]:

# Setup the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
print(f'Device is set to : {device}')


Device is set to : cuda


In [3]:

# Set up the transforms and train/test loaders
image_size = 192

mnist_transform_train = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=2),
        transforms.Resize((image_size, image_size)), 
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

mnist_transform_test = transforms.Compose([
        transforms.Resize((image_size, image_size)), 
        transforms.Grayscale(num_output_channels=3), 
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])


train_loader, test_loader = get_mnist_dataloaders(
                                    data_dir = '../datasets',
                                    batch_size = 2,
                                    image_size = 192,
                                    transform_train = mnist_transform_train ,
                                    transform_test = mnist_transform_test)

In [13]:
for x,y in train_loader:
    break
x.shape

torch.Size([2, 3, 192, 192])

In [14]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.grad)


classifier.0.b tensor([ 0.0000,  0.0000,  0.5000,  0.0000,  0.5000,  0.0000, -0.5000, -0.5000,
         0.0000,  0.0000], device='cuda:0')
classifier.0.core tensor([[[[-1.1547e+02, -2.7374e+02, -3.1441e+02,  ..., -3.4655e+02,
            1.3253e+02,  7.0446e+01],
          [ 8.6427e+02, -4.9450e+02,  4.2574e+02,  ...,  2.8714e+02,
           -1.5102e+02,  6.7230e+02],
          [-3.6247e+02,  4.6636e+02,  1.7716e+01,  ...,  1.1445e+02,
           -2.2293e+01, -4.0410e+02]],

         [[-2.0746e+02,  7.3540e+01, -1.3642e+02,  ..., -1.0988e+02,
            5.1181e+01, -1.4008e+02],
          [-9.7488e+01,  6.3345e+02,  3.8978e+02,  ...,  4.9152e+02,
           -1.7397e+02, -3.4830e+02],
          [-1.9085e+02, -1.8209e+00, -1.7815e+02,  ..., -1.6409e+02,
            7.0057e+01, -9.6100e+01]],

         [[ 1.7374e+02,  2.9977e+02,  3.8811e+02,  ...,  4.1975e+02,
           -1.6234e+02, -5.3127e+01],
          [-1.4840e+02, -1.3506e+02, -2.3981e+02,  ..., -2.4879e+02,
            9.8660e+0

In [4]:
for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        print(name, "Gradient on CUDA:", param.grad.is_cuda)


NameError: name 'model' is not defined

In [7]:
# Set up the new classifier 

new_classifier = nn.Sequential(
    TRL(input_size=(2,2048,6,6), output=(10), rank=(200,3,3,8),ignore_modes = (0,), bias = True, device = device).to(device)
)

# Set up the model, optimizer and criterion
model = Resnet50(pretrained=False,
                        weights_path='../weights/resnet50_weights.pth',
                        tensorized=True,
                        input_shape=(192,192),
                        num_classes=10,
                        avg_pool=False,
                        new_classifier=new_classifier).to(device)

# Load pretrained from Tests


num_parameters = count_parameters(model)
classifier_parameters = count_parameters(model.classifier)
print(f'This Model has {num_parameters} parameters')
print(f'This Model has {classifier_parameters} classifier parameters')



criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

NameError: name 'device' is not defined

In [7]:
model(x.to(device))

NameError: name 'x' is not defined

In [8]:
# Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)
    
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
                


    
        loss.backward()
        for name, param in model.named_parameters():
            if param.requires_grad and param.grad is not None:
                print(name, "Gradient on CUDA:", param.grad.is_cuda)
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'Train epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [9]:
# Set up the directories to save the results
TEST_ID = 'Test_ID091'
result_dir = os.path.join('../results', TEST_ID)
result_subdir = os.path.join(result_dir, 'accuracy_stats')
model_subdir = os.path.join(result_dir, 'model_stats')

os.makedirs(result_subdir, exist_ok=True)
os.makedirs(model_subdir, exist_ok=True)

with open(os.path.join(result_dir, 'model_stats', 'model_info.txt'), 'a') as f:
    f.write(f'total number of parameters:\n{num_parameters}\ntotal number of classifier parameters:\n{classifier_parameters}')

# Freeze Convolutional Layers
layer = 0
for child in model.children():
    layer+=1
    if layer < 3:
        for param in child.parameters():
            param.requires_grad = False

# Train and Test The Model - Frozen Layers
n_epoch = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)

    report = report_train + '\n' + report_test + '\n\n'
    if epoch % 10 == 0:
        model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        torch.save(model.state_dict(), model_path)
    with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
        f.write(report)
        
# Unfreeze all layers
layer = 0
for child in model.children():
    layer+=1
    if layer < 3:
        for param in child.parameters():
            param.requires_grad = True
            
# Train and Test The Model - Unfrozen Layers - comment if not required
n_epoch_additional = 1
print(f'Training for Additional {len(range(n_epoch_additional))} epochs\n')
for epoch in range(n_epoch+1,n_epoch+n_epoch_additional+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)

    report = report_train + '\n' + report_test + '\n\n'
    if epoch % 5 == 0:
        model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        torch.save(model.state_dict(), model_path)
    with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
        f.write(report)

Training for 1 epochs

classifier.0.b Gradient on CUDA: True
classifier.0.core Gradient on CUDA: True
classifier.0.factors.0 Gradient on CUDA: False
classifier.0.factors.1 Gradient on CUDA: False
classifier.0.factors.2 Gradient on CUDA: False
classifier.0.factors.3 Gradient on CUDA: False


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!